In [36]:
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model_name="gpt-5-mini", temperature=0)

## Creating Tools

In [13]:
from langchain.tools import tool

In [ ]:
@tool
def duckduckgo_search(query) -> str:
    """
    Use this function to perform a search & to answer questions that require up-to-date information. 
    this is a type of web search tool that can be used to retrieve information from the web.
    """
    from langchain_community.tools import DuckDuckGoSearchRun
    search = DuckDuckGoSearchRun()
    return search.invoke(query)

# duckduckgo_search("What is the capital of France?")


In [17]:
duckduckgo_search.invoke("What is the capital of France?")

'1 week ago - Paris is the capital and largest city of France, with an estimated city population of 2.04 million in an area of 105.4 km2 (40.7 sq mi), as of January 2026, and a metropolitan population of 13.3 million (2023). Located on the Seine in the centre of the Île-de-France region, it is the largest ... 1 month ago - This is a chronological list of capitals of France. The capital of France has been Paris since its liberation in 1944. 2 days ago - Its metropolitan area extends from ... and the North Sea. Its 18 integral regions—five of which are overseas—span a combined area of 632,702 km2 (244,288 sq mi), with a total population estimated at over 69.1 million in 2026. Its capital, largest city and main cultural and economic centre is Paris, with a metropolitan population of over 13 million. Metropolitan France was settled ... 2 weeks ago - The capital of France is Paris. January 25, 2026 - Paris is the capital of France, and it feels a bit like a beautifully organized bookshelf: 

In [22]:
@tool
def wikipedia_search(query) -> str:
    """
    Use this function to perform a search & to answer questions related to person or a place.
    """
    from langchain_community.tools import WikipediaQueryRun
    from langchain_community.utilities import WikipediaAPIWrapper

    wiki = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
    return wiki.invoke(query)

In [24]:
wikipedia_search.invoke("Sam Altman")

'Page: Sam Altman\nSummary: Samuel Harris Altman (born April 22, 1985) is an American entrepreneur and investor who has been the chief executive officer (CEO) of the artificial intelligence company OpenAI since 2019.\nAltman attended Stanford University for two years before he dropped out and co-founded Loopt, a geosocial networking application for smartphones. Loopt was acquired by Green Dot Corporation for $43.4 million in March 2012. In 2011, Altman joined Y Combinator, a technology startup accelerator and venture capital firm, and was the company\'s president from 2014 to 2019. He is a billionaire, due to investments in over 400 companies including Reddit, Worldcoin, Helion Energy and Instacart.\nAltman co-founded OpenAI in 2015 and became its CEO in 2019, a role that made him a prominent figure of the AI boom. He supervised the launch of ChatGPT in November 2022. In 2023, he was ousted by the organization\'s board of directors for not being "consistently candid". The move was met 

In [34]:
@tool
def arxiv_search(query: str) -> str:
    """Search arXiv for recent research papers matching the query."""
    import arxiv

    search = arxiv.Search(
        query=query,
        max_results=5,
        sort_by=arxiv.SortCriterion.SubmittedDate,
        sort_order=arxiv.SortOrder.Descending,
    )
    papers = list(arxiv.Client().results(search))
    if not papers:
        return f"No arXiv papers found for: {query}"

    return "\n\n".join(
        f"Title: {paper.title}\n"
        f"Authors: {', '.join(author.name for author in paper.authors)}\n"
        f"Published: {paper.published:%Y-%m-%d}\n"
        f"Summary: {paper.summary}\n"
        f"URL: {paper.entry_id}"
        for paper in papers
    )

In [35]:
arxiv_search.invoke("What is the latest research on AI?")

"Title: Telescopic Language Models\nAuthors: Zhilin Guo, Boqiao Zhang, Hakan Aktas, Kyle Fogarty, Nursena Koprucu Aslan, Wenzhao Li, Canberk Baykal, Albert Miao, Siyu Hong, Yixiao Liu, Adam Wu, Ashish Kumar Singh, Sakar Khattar, Chenliang Zhou, Weihao Xia, Cristina Nader Vasconcelos, Cengiz Oztireli\nPublished: 2026-09-28\nSummary: One deployed language model must often serve many compute budgets, yet serving each budget still means a separate training or compression run per point. We train a Telescopic Language Model (TLM) to be that continuum: a nested-capacity Transformer supervised by stochastic prefix supervision with a full anchor. At every step, one randomly truncated prefix of the capacity axis is trained against the full next-token target, alongside one full-capacity pass, so the trained artifact is a valid language model at every depth. Two forward-backward passes per step, no architectural change, nothing extra at inference. Fixed-exit suites such as Matryoshka Language Mode

## Binding tools

In [38]:
## Binding the tools to the LLM

#create a list of tools to bind to the LLM
tool_kit = [duckduckgo_search, wikipedia_search, arxiv_search]
#bind the tools to the LLM
llm_with_tools = llm.bind_tools(tool_kit)

In [42]:
llm_with_tools.invoke("Sam Altman is the CEO of which company?, make use of the tools")

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 231, 'total_tokens': 263, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-ETVLdF30rLR7abujcKbAcEEab5FZp', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0ee14-8fce-7620-a36d-461f89ca5159-0', tool_calls=[{'name': 'duckduckgo_search', 'args': {'query': 'Sam Altman CEO of which company'}, 'id': 'call_Z5REx1tn9NtDiDP5KCWTJv5d', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 231, 'output_tokens': 32, 'total_tokens': 263, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_det

In [49]:
## Creating a React Agent with the LLM and the tools
from langchain.agents import create_agent
agent = create_agent(llm_with_tools, tool_kit)

# Call the agent with a query
agent.invoke({"messages": [{"role": "user", "content": "Sam Altman is the CEO of which company?, make use of the tools"}]})

{'messages': [HumanMessage(content='Sam Altman is the CEO of which company?, make use of the tools', additional_kwargs={}, response_metadata={}, id='2e4e714d-e17d-4678-b11d-8fceb25d1647'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 153, 'prompt_tokens': 231, 'total_tokens': 384, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-ETVvmd4GXJEr2kEAemBnis8lbJKro', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0ee36-c24d-7c33-bc16-d8c70780f851-0', tool_calls=[{'name': 'wikipedia_search', 'args': {'query': 'Sam Altman'}, 'id': 'call_5RZs2TZstnEg6JlCLt8hUdTM', 'type': 'tool_call'}], i